<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/09-gradient-boosting.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 9 — Gradient Boosting

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/gradient-boosting/).

Fitting the residual: small trees added one at a time, each correcting what the sum so far gets wrong.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

def bicycle(km):
    return 10 + 3.2 * km + 0.15 * km**2

def rush(hour):
    return 25 + 12 * np.exp(-(hour - 13)**2 / 2) + 15 * np.exp(-(hour - 20)**2 / 2)

rng = np.random.default_rng(1)
n = 300
km = rng.uniform(0.5, 8.0, n)
hour = rng.uniform(10, 23, n)
minutes = bicycle(km) + rush(hour) - 25 + rng.normal(0, 4.0, n)      # distance plus a lunch and dinner rush
X = np.column_stack([km, hour])

km_t = rng.uniform(0.5, 8.0, 2000); hour_t = rng.uniform(10, 23, 2000)
min_t = bicycle(km_t) + rush(hour_t) - 25 + rng.normal(0, 4.0, 2000)
X_test = np.column_stack([km_t, hour_t])

def mae(a, b): return np.mean(np.abs(a - b))
print(f'{n} orders. Always guessing the average: {mae(np.full(2000, minutes.mean()), min_t):.2f} min on new orders')
print(f'the true formula: {mae(bicycle(km_t) + rush(hour_t) - 25, min_t):.2f} min')

## 1. Fit the residual, by hand

In [ ]:
prediction = np.full(n, minutes.mean())                 # round 0: the average
print(f'{"round":>5} {"train MAE":>10} {"test MAE":>9}')
test_prediction = np.full(2000, minutes.mean())
for r in range(1, 6):
    residual = minutes - prediction                    # what the sum so far gets wrong
    stump = DecisionTreeRegressor(max_depth=2).fit(X, residual)
    prediction = prediction + stump.predict(X)
    test_prediction = test_prediction + stump.predict(X_test)
    print(f'{r:>5} {mae(prediction, minutes):>10.2f} {mae(test_prediction, min_t):>9.2f}')

## 2. Smaller steps, more of them

In [ ]:
def boost(X, y, rounds, rate, depth=2, X_eval=None, y_eval=None, loss='squared'):
    pred = np.full(len(y), y.mean())
    eval_pred = None if X_eval is None else np.full(len(y_eval), y.mean())
    history = []
    for r in range(rounds):
        if loss == 'squared':
            target = y - pred                              # negative gradient of squared error
        else:
            target = np.sign(y - pred)                     # negative gradient of absolute error
        tree = DecisionTreeRegressor(max_depth=depth, random_state=r).fit(X, target)
        pred = pred + rate * tree.predict(X)
        if X_eval is not None:
            eval_pred = eval_pred + rate * tree.predict(X_eval)
            history.append(mae(eval_pred, y_eval))
    return pred, eval_pred, history

print(f'{"rate":>5} {"rounds":>7} {"test MAE":>9}')
for rate, rounds in [(1.0, 10), (1.0, 50), (0.1, 50), (0.1, 200), (0.1, 1000), (0.03, 1000)]:
    _, test_pred, _ = boost(X, minutes, rounds, rate, X_eval=X_test, y_eval=min_t)
    print(f'{rate:>5} {rounds:>7} {mae(test_pred, min_t):>9.2f}')

## 3. Too many rounds, and when to stop

In [ ]:
_, _, history = boost(X, minutes, 2000, 0.1, X_eval=X_test, y_eval=min_t)
best = int(np.argmin(history))
print(f'test MAE after 50 rounds {history[49]:.2f}, after 200 {history[199]:.2f}, after 2000 {history[-1]:.2f}')
print(f'lowest at round {best + 1}: {history[best]:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(np.arange(1, 2001), history)
ax.axvline(best + 1, color='black', ls='--', lw=1, label=f'best, round {best + 1}')
ax.set_xlabel('boosting rounds'); ax.set_ylabel('MAE on held-out orders'); ax.set_xscale('log'); ax.legend()
plt.show()

## 4. Early stopping, done without the test set

In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(X, minutes, test_size=0.25, random_state=0)
_, _, val_history = boost(X_fit, y_fit, 2000, 0.1, X_eval=X_val, y_eval=y_val)
stop = int(np.argmin(val_history)) + 1
_, test_pred, _ = boost(X_fit, y_fit, stop, 0.1, X_eval=X_test, y_eval=min_t)
print(f'validation set says stop at round {stop}; that model scores {mae(test_pred, min_t):.2f} min on the test set')

## 5. Why "gradient": a different loss, the same machine

In [ ]:
minutes_o = minutes.copy()
wild = rng.choice(n, 6, replace=False)
minutes_o[wild] += 90                                     # six orders stuck in lifts

for loss in ['squared', 'absolute']:
    _, test_pred, _ = boost(X, minutes_o, 300, 0.1, X_eval=X_test, y_eval=min_t, loss=loss)
    print(f'{loss:<9} loss, trained with six wild orders: {mae(test_pred, min_t):.2f} min on ordinary new orders')

## 6. Depth of each tree

In [ ]:
print(f'{"depth":>5} {"test MAE":>9}')
for depth in [1, 2, 3, 5, 8]:
    _, test_pred, _ = boost(X, minutes, 300, 0.1, depth=depth, X_eval=X_test, y_eval=min_t)
    print(f'{depth:>5} {mae(test_pred, min_t):>9.2f}')

## 7. The one line you would ship

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=2, random_state=0).fit(X, minutes)
rf = RandomForestRegressor(n_estimators=300, random_state=0).fit(X, minutes)
lin = LinearRegression().fit(X, minutes)
for name, m in [('linear regression', lin), ('random forest', rf), ('gradient boosting', gbr)]:
    print(f'{name:<18} {mae(m.predict(X_test), min_t):.2f} min on new orders')

## 8. Boosting a classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

def make_loans(n, seed=0):
    rng = np.random.default_rng(seed)
    income = rng.uniform(15, 120, n); debt_ratio = rng.uniform(0.0, 0.8, n)
    missed = rng.integers(0, 4, n); age = rng.uniform(21, 70, n)
    risky = ((income < 40) & (debt_ratio > 0.4)) | (missed >= 2)
    default = np.where(risky, rng.uniform(size=n) < 0.7, rng.uniform(size=n) < 0.05).astype(int)
    return np.column_stack([income, debt_ratio, missed, age]), default

Xl, yl = make_loans(600, seed=1)
Xl_t, yl_t = make_loans(5000, seed=2)
for name, m in [('logistic regression', LogisticRegression(max_iter=1000)),
                ('random forest', RandomForestClassifier(n_estimators=300, random_state=0)),
                ('gradient boosting', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=0))]:
    m.fit(Xl, yl)
    print(f'{name:<20} accuracy {m.score(Xl_t, yl_t):.3f}')

## 9. Still a tree underneath

In [ ]:
print(f'16 km at 3pm: truth {bicycle(16) + rush(15) - 25:.1f} min, boosting says {gbr.predict([[16, 15]])[0]:.1f}, linear says {lin.predict([[16, 15]])[0]:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
h = np.linspace(10, 23, 300)
axes[0].plot(h, rush(h) - 25 + bicycle(4), color='black', lw=1, ls='--', label='truth at 4 km')
axes[0].plot(h, gbr.predict(np.column_stack([np.full(300, 4.0), h])), lw=2, label='boosting at 4 km')
axes[0].plot(h, lin.predict(np.column_stack([np.full(300, 4.0), h])), lw=1, label='linear at 4 km')
axes[0].set_xlabel('hour of day'); axes[0].set_ylabel('minutes'); axes[0].legend()
axes[1].plot(np.arange(1, 2001), history)
axes[1].set_xscale('log'); axes[1].set_xlabel('rounds'); axes[1].set_ylabel('held-out MAE'); axes[1].set_title('more rounds, then too many')
plt.tight_layout(); plt.show()

## Try it yourself

1. In section 2, try `rate=1.0` with 1,000 rounds. Where does the test error end up, and why is this worse than 0.1 for 1,000?
2. Change the six wild orders to sixty. Does absolute loss still protect the model?
3. In section 8, give the boosted classifier `max_depth=6`. Does it beat depth 2? Check with cross-validation, since the test set is for the final number only.